In [52]:
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [53]:
df = pd.read_csv(
    "C:/Users/sathi/Downloads/monster_com-job_sample.csv",
    engine="python",
    on_bad_lines="skip"
)

# Keep only necessary columns
df = df[['job_title', 'job_description', 'organization', 'location']]

# Drop null values
df = df.dropna(subset=['job_description'])

# Remove short/invalid descriptions
df = df[df['job_description'].str.len() > 300]

# Remove duplicates
df = df.drop_duplicates(subset='job_description')

df = df.reset_index(drop=True)

print("Clean dataset size:", df.shape)

Clean dataset size: (327, 4)


In [7]:
# # engine="python" → Uses Python engine (better for messy CSV files).

# By default, pandas uses the C engine (faster).
# engine="python" forces it to use the Python engine.

# Why use it?

# Handles complex separators

# Better for irregular formatting

# Needed when using on_bad_lines
# # quoting=3 → Ignores quotes (csv.QUOTE_NONE).
# Normally pandas understands that "Hello, world" is one value.

# With quoting=3

# Quotes are ignored — they are treated as normal characters.

# # on_bad_lines="skip" → Skips rows with errors instead of stopping.


# name,age
# John,25
# Anna,30,ExtraValue
# Mike,22
##output
#    name  age
# 0  John   25
# 1  Mike   22


In [54]:
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(str(text).lower())
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and len(token.text) > 2
    ]
    return " ".join(tokens)

In [49]:
# # Clean Job Descriptions
# df['clean_description'] = df['job_description'].apply(preprocess)

In [55]:
df['combined_text'] = df['job_title'] + " " + df['job_description']
df['clean_text'] = df['combined_text'].apply(preprocess)

In [56]:
# Load Sentence Transformer Model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
job_embeddings = model.encode(
    df['clean_text'].tolist(),
    show_progress_bar=True
)

# Save embeddings for reuse
np.save("job_embeddings.npy", job_embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [26]:
df.to_pickle("job_data.pkl")
np.save("job_embeddings.npy", job_embeddings)

print("Files saved successfully!")

In [39]:
resume_text = """
Mechanical Engineer experienced in CAD design,
thermodynamics, manufacturing processes,
and project management.
"""

In [40]:
# Clean Resume
clean_resume = preprocess(resume_text)
resume_embedding = model.encode([clean_resume])

In [41]:
# Compute Similarity
similarities = cosine_similarity(
    resume_embedding,
    job_embeddings
)[0]

In [42]:
similarities = similarities * 100

In [43]:
df['match_score'] = similarities

In [44]:
top_jobs = df.sort_values(
    by='match_score',
    ascending=False
).head(5)

top_jobs[['job_title', 'organization', 'location', 'match_score']]

,job_title,organization,location,match_score
130,Mechanical/Machine Designer – Nashville,Manufacturing - Other,"Nashville, TN 37203",55.254330
173,Process Engineer Job in Auburn,Automotive and Parts Mfg,"Auburn, IN 46706",52.679909
95,Mechanical Engineer Relocate to Decatur,NaN,"Bloomington, IL",51.011265
226,Designer II Job in Amarillo,NaN,System One is currently seeking a Designer II ...,50.040863
236,Packaging Engineer Job in Chicago,"Chicago, IL 60661",Packaging Engineer,49.011070
